# EFDP — Universal Data Cleaning Template

**Project:** UC20 Enterprise Financial Data Platform  
**Purpose:** Reusable, audit-friendly cleaning workflow for PostgreSQL staging tables using Python, Pandas, and SQLAlchemy.

## Important safeguards
- Reads from `stg_*` tables and writes to a separate cleaned layer (`clean_*` by default).
- Never updates or deletes rows/columns in the staging tables.
- Defaults to dry-run mode; no database writes until explicitly enabled.
- Does not invent missing values or automatically remove business-key duplicates.
- Exact duplicate removal, column exclusion, and NULL quarantine are configurable and opt-in.
- Records changes and exports before/after evidence for Sprint 2.

### Workflow
1. Configure database and table.
2. Load staging data and capture baseline profile.
3. Standardize text conservatively.
4. Optionally exclude approved columns and remove exact duplicate rows.
5. Report duplicate business keys and apply explicit NULL policies.
6. Validate output and export audit evidence.
7. Optionally write cleaned rows to a new target table.

## 1. Install dependencies (if needed)

Run once in your environment:
```python
%pip install pandas sqlalchemy psycopg2-binary
```


In [1]:
import os
import re
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text, inspect

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Pandas:", pd.__version__)


Pandas: 3.0.6


## 2. Database connection and universal cleaning configuration

This cell connects to PostgreSQL on your Windows laptop and defines the reusable cleaning controls. Keep `DRY_RUN = True` and `WRITE_TO_DATABASE = False` while reviewing results.

- `COLUMNS_TO_DROP`: explicitly approved columns to exclude from the cleaned output only.
- `DROP_EXACT_DUPLICATES`: removes identical rows from the in-memory cleaned DataFrame only when set to `True`.
- `KEY_COLUMNS`: reports repeated business keys; these rows are **not** automatically deleted.
- `NULL_HANDLING_RULES`: `retain`, `flag`, or `quarantine` for selected columns. No default values are invented.
- `MANDATORY_COLUMNS`: reports missing mandatory values. Use rules approved by your project/data owner.

The original `stg_*` table is never updated or dropped by this notebook.

In [2]:
import getpass
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL

DB_USER = "postgres"
DB_PASSWORD = getpass.getpass("PostgreSQL password: ")
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "new_cleaning"

connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as conn:
    print("Connected to database:", conn.execute(
        text("SELECT current_database()")
    ).scalar())

# -------- Universal cleaning configuration --------
SCHEMA = "public"
SOURCE_TABLE = "stg_bseg"       # Change for each table
TARGET_TABLE = "clean_bseg"     # Separate cleaned-layer table

DRY_RUN = False                  # Keep True while reviewing results
WRITE_TO_DATABASE = True       # Enable only after validation

TRIM_TEXT_WHITESPACE = True
NORMALIZE_EMPTY_STRINGS_TO_NULL = True
CASE_NORMALIZATION = {}         # e.g. {"country_code": "upper"} only if approved
COLUMN_RENAMES = {}             # approved source_name: target_name mapping
COLUMNS_TO_DROP = ["tax_code",]            # e.g. ["chart_of_accounts"] only after review
DATE_COLUMNS = []               # parse only after checking source formats
KEY_COLUMNS = []                # confirmed business key columns only
MANDATORY_COLUMNS = []          # confirmed mandatory fields only

# Duplicate policy:
DROP_EXACT_DUPLICATES = True   # Enable only after investigation/approval
# Duplicate business keys are reported, never automatically deleted by this template.

# NULL policy: for each configured column choose "retain", "flag", or "quarantine".
# "quarantine" moves rows with NULL in that column out of df_clean into df_quarantine.
# No automatic imputation/default values are applied.
NULL_HANDLING_RULES = {}        # e.g. {"customer_name": "quarantine"}

MAX_SAMPLE_ISSUES = 100

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError("Invalid schema identifier")
for name in [SOURCE_TABLE, TARGET_TABLE]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid table identifier: {name}")
if WRITE_TO_DATABASE and DRY_RUN:
    raise ValueError("Set DRY_RUN=False before enabling database writes.")
if WRITE_TO_DATABASE and TARGET_TABLE == SOURCE_TABLE:
    raise ValueError("Target table must be different from the staging source table.")
if any(action not in {"retain", "flag", "quarantine"} for action in NULL_HANDLING_RULES.values()):
    raise ValueError("NULL_HANDLING_RULES actions must be retain, flag, or quarantine.")

print({
    "schema": SCHEMA,
    "source": SOURCE_TABLE,
    "target": TARGET_TABLE,
    "dry_run": DRY_RUN,
    "write_enabled": WRITE_TO_DATABASE,
    "drop_columns": COLUMNS_TO_DROP,
    "drop_exact_duplicates": DROP_EXACT_DUPLICATES,
    "null_rules": NULL_HANDLING_RULES
})

Connected to database: new_cleaning
{'schema': 'public', 'source': 'stg_bseg', 'target': 'clean_bseg', 'dry_run': False, 'write_enabled': True, 'drop_columns': ['tax_code'], 'drop_exact_duplicates': True, 'null_rules': {}}


## 3. Connect and load the source table

The database connection is already created in Section 2. This section reuses that same `engine`; it does not create another connection.

In [3]:
inspector = inspect(engine)
if not inspector.has_table(SOURCE_TABLE, schema=SCHEMA):
    raise ValueError(f"Source table {SCHEMA}.{SOURCE_TABLE} does not exist.")

source_sql = f'SELECT * FROM "{SCHEMA}"."{SOURCE_TABLE}"'
df_raw = pd.read_sql_query(text(source_sql), engine)

print("Loaded rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
display(df_raw.head())

Loaded rows: 332106
Columns: 21


,client_id,company_code,accounting_document_no,fiscal_year,line_item_no,account_type,debit_credit_indicator,amount_local_currency,amount_document_currency,assignment_no,line_item_text,payment_terms,cost_center_id,gl_account_no,customer_id,vendor_id,payement_baseline_date,tax_code,clearing_document_number,clearing_date,special_gl_indicator
0,100,C003,5105600151,2022,2,S,S,300.0,300.0,450000003500010,NaN,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
1,50,C003,5105600143,2022,2,S,S,600.0,600.0,450000003400010,NaN,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
2,100,C003,5105600152,2022,2,S,S,550.0,550.0,450000003600010,NaN,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
3,100,C003,5105600153,2022,2,S,S,360.0,360.0,450000003900010,NaN,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
4,100,C003,5105600154,2022,2,S,S,480.0,480.0,450000004000010,NaN,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN


## 4. Baseline profiling

Capture nulls, exact duplicates, dtypes, and basic table-level counts before cleaning.


In [4]:
def profile_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "null_count": frame.isna().sum(),
        "null_pct": (frame.isna().mean() * 100).round(3),
        "distinct_non_null": frame.nunique(dropna=True),
        "blank_string_count": frame.apply(
            lambda s: (s.astype("string").str.strip().eq("").fillna(False)).sum()
            if (pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s))
            else 0
        )
    }).sort_values(["null_pct", "null_count"], ascending=False)

baseline_profile = profile_dataframe(df_raw)
baseline_summary = {
    "source_table": SOURCE_TABLE,
    "row_count": int(len(df_raw)),
    "column_count": int(len(df_raw.columns)),
    "exact_duplicate_rows_excess": int(df_raw.duplicated().sum()),
    "profiled_at_utc": datetime.now(timezone.utc).isoformat()
}
display(pd.DataFrame([baseline_summary]))
display(baseline_profile)


,source_table,row_count,column_count,exact_duplicate_rows_excess,profiled_at_utc
0,stg_bseg,332106,21,77,2026-10-22T16:29:53.195256+00:00


,dtype,null_count,null_pct,distinct_non_null,blank_string_count
tax_code,object,332106,100.000,0,0
special_gl_indicator,str,332101,99.998,1,0
line_item_text,str,332054,99.984,1,0
cost_center_id,str,332026,99.976,12,0
clearing_document_number,str,328652,98.960,1,0
clearing_date,str,328652,98.960,1,0
customer_id,str,293496,88.374,1,0
payment_terms,str,258029,77.695,1,0
payement_baseline_date,str,257694,77.594,1489,0
vendor_id,str,144799,43.600,28,0


## 5. Conservative standardization

This step only trims surrounding whitespace in text values and optionally converts empty/whitespace-only strings to NULL. Both can be disabled in configuration. It does not change case, identifiers, amounts, dates, or missing values beyond the explicitly configured empty-string rule.


In [5]:
df_clean = df_raw.copy(deep=True)
change_log = []
issue_rows = []
df_quarantine = pd.DataFrame(columns=df_clean.columns)
quarantine_mask = pd.Series(False, index=df_clean.index)

# Conservative text normalization
for col in df_clean.columns:
    is_text = pd.api.types.is_object_dtype(df_clean[col]) or pd.api.types.is_string_dtype(df_clean[col])
    if not is_text:
        continue

    before = df_clean[col].copy()
    if TRIM_TEXT_WHITESPACE:
        df_clean[col] = df_clean[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    if NORMALIZE_EMPTY_STRINGS_TO_NULL:
        df_clean[col] = df_clean[col].map(
            lambda x: pd.NA if isinstance(x, str) and x == "" else x
        )

    changed = int((before.fillna("__NA__").astype(str) != df_clean[col].fillna("__NA__").astype(str)).sum())
    if changed:
        change_log.append({
            "column": col,
            "rule": "trim whitespace / normalize empty strings",
            "rows_affected": changed,
            "reason": "Conservative text normalization"
        })

# Explicitly configured column drops affect df_clean only, not PostgreSQL staging.
missing_drop_columns = [col for col in COLUMNS_TO_DROP if col not in df_clean.columns]
if missing_drop_columns:
    raise ValueError(f"Configured drop columns not found: {missing_drop_columns}")
for col in COLUMNS_TO_DROP:
    change_log.append({
        "column": col,
        "rule": "exclude column from cleaned output",
        "rows_affected": int(df_clean[col].notna().sum()),
        "reason": "Explicitly approved in COLUMNS_TO_DROP"
    })
if COLUMNS_TO_DROP:
    df_clean = df_clean.drop(columns=COLUMNS_TO_DROP)

# Exact duplicate removal is opt-in and occurs after configured text normalization.
exact_duplicates_before = int(df_clean.duplicated().sum())
if DROP_EXACT_DUPLICATES:
    df_clean = df_clean.drop_duplicates(keep="first").copy()
    removed = exact_duplicates_before
    change_log.append({
        "column": "*",
        "rule": "drop exact duplicate rows",
        "rows_affected": removed,
        "reason": "Explicitly enabled by DROP_EXACT_DUPLICATES; retained first identical row"
    })

print("Standardization complete.")
print("Exact duplicate excess after standardization/removal:", int(df_clean.duplicated().sum()))
display(pd.DataFrame(change_log, columns=["column", "rule", "rows_affected", "reason"]))

Standardization complete.
Exact duplicate excess after standardization/removal: 0


,column,rule,rows_affected,reason
0,assignment_no,trim whitespace / normalize empty strings,24,Conservative text normalization
1,tax_code,exclude column from cleaned output,0,Explicitly approved in COLUMNS_TO_DROP
2,*,drop exact duplicate rows,77,Explicitly enabled by DROP_EXACT_DUPLICATES; retained first identical row


## 6. Optional approved column renaming and case normalization

Only configure these after confirming the desired target-layer naming convention and source semantics. Renaming creates a cleaned-layer column name; it does not alter the staging table.


In [6]:
if COLUMN_RENAMES:
    missing_rename_sources = [c for c in COLUMN_RENAMES if c not in df_clean.columns]
    if missing_rename_sources:
        raise ValueError(f"Rename source columns not found: {missing_rename_sources}")
    if set(COLUMN_RENAMES.values()) & (set(df_clean.columns) - set(COLUMN_RENAMES.keys())):
        raise ValueError("A rename target would collide with an existing column.")
    df_clean = df_clean.rename(columns=COLUMN_RENAMES)
    for old, new in COLUMN_RENAMES.items():
        change_log.append({
            "column": old,
            "rule": f"rename to {new}",
            "rows_affected": len(df_clean),
            "reason": "Approved naming standardization"
        })

for col, mode in CASE_NORMALIZATION.items():
    if col not in df_clean.columns:
        raise ValueError(f"Configured case-normalization column not found: {col}")
    if mode not in {"upper", "lower"}:
        raise ValueError(f"Unsupported case mode for {col}: {mode}")
    fn = str.upper if mode == "upper" else str.lower
    mask = df_clean[col].map(lambda x: isinstance(x, str))
    df_clean.loc[mask, col] = df_clean.loc[mask, col].map(fn)

display(pd.DataFrame(change_log, columns=["column", "rule", "rows_affected", "reason"]))


,column,rule,rows_affected,reason
0,assignment_no,trim whitespace / normalize empty strings,24,Conservative text normalization
1,tax_code,exclude column from cleaned output,0,Explicitly approved in COLUMNS_TO_DROP
2,*,drop exact duplicate rows,77,Explicitly enabled by DROP_EXACT_DUPLICATES; retained first identical row


## 7. Date parsing — validate before converting

Set `DATE_COLUMNS` only after reviewing source formats. Invalid non-null values are reported; they are not silently dropped. A parsed copy is used in the cleaned layer.


In [7]:
date_issues = []
for col in DATE_COLUMNS:
    if col not in df_clean.columns:
        raise ValueError(f"Configured date column not found: {col}")
    original = df_clean[col]
    parsed = pd.to_datetime(original, errors="coerce")
    invalid_mask = original.notna() & parsed.isna()
    if invalid_mask.any():
        for idx, value in original[invalid_mask].head(MAX_SAMPLE_ISSUES).items():
            date_issues.append({
                "row_index": str(idx),
                "column": col,
                "original_value": str(value),
                "issue": "Non-null value could not be parsed as date/time"
            })
    df_clean[col] = parsed

display(pd.DataFrame(date_issues, columns=["row_index", "column", "original_value", "issue"]))


,row_index,column,original_value,issue


## 8. Duplicate analysis — do not delete by default

Exact duplicate rows are counted separately from duplicate business keys. Business keys must be confirmed from the data dictionary/source owners. If duplicates are found, investigate whether they represent repeated ingestion, legitimate history, or conflicting records before enabling a removal rule.


In [8]:
duplicate_summary = {
    "exact_duplicate_rows_excess_in_source": int(df_raw.duplicated().sum()),
    "exact_duplicate_rows_excess_after_standardization": int(exact_duplicates_before),
    "exact_duplicate_rows_excess_remaining": int(df_clean.duplicated().sum()),
    "exact_duplicates_removed": int(exact_duplicates_before if DROP_EXACT_DUPLICATES else 0),
    "configured_key_columns": KEY_COLUMNS
}

if KEY_COLUMNS:
    # Key columns are evaluated against the post-rename cleaned column names.
    missing_keys = [c for c in KEY_COLUMNS if c not in df_clean.columns]
    if missing_keys:
        raise ValueError(f"Configured key columns not found: {missing_keys}")
    duplicate_key_mask = df_clean.duplicated(subset=KEY_COLUMNS, keep=False)
    null_key_mask = df_clean[KEY_COLUMNS].isna().any(axis=1)
    duplicate_summary["rows_in_duplicate_key_groups"] = int(duplicate_key_mask.sum())
    duplicate_summary["rows_with_null_key_component"] = int(null_key_mask.sum())
    key_samples = df_clean.loc[duplicate_key_mask | null_key_mask].head(MAX_SAMPLE_ISSUES)
else:
    duplicate_summary["rows_in_duplicate_key_groups"] = None
    duplicate_summary["rows_with_null_key_component"] = None
    key_samples = pd.DataFrame()

display(pd.DataFrame([duplicate_summary]))
display(key_samples)

,exact_duplicate_rows_excess_in_source,exact_duplicate_rows_excess_after_standardization,exact_duplicate_rows_excess_remaining,exact_duplicates_removed,configured_key_columns,rows_in_duplicate_key_groups,rows_with_null_key_component
0,77,77,0,77,[],None,None


""


## 9. Business-rule checks and issue register

Add table-specific checks only when the rule is supported by the data dictionary or approved by the data owner. This example flags missing values in configured mandatory columns; it does not fill them or delete rows.


In [9]:
# Mandatory field checks: report missing values; do not impute them.
mandatory_issues = []
for col in MANDATORY_COLUMNS:
    if col not in df_clean.columns:
        raise ValueError(f"Mandatory column not found: {col}")
    mask = df_clean[col].isna()
    for idx in df_clean.index[mask][:MAX_SAMPLE_ISSUES]:
        mandatory_issues.append({
            "row_index": str(idx),
            "column": col,
            "issue": "Missing mandatory value",
            "action": "Review; no automatic imputation"
        })

# Apply explicit per-column NULL policies.
null_rule_issues = []
for col, action in NULL_HANDLING_RULES.items():
    if col not in df_clean.columns:
        raise ValueError(f"NULL rule column not found: {col}")
    mask = df_clean[col].isna()
    count = int(mask.sum())
    if action in {"flag", "quarantine"}:
        for idx in df_clean.index[mask][:MAX_SAMPLE_ISSUES]:
            null_rule_issues.append({
                "row_index": str(idx),
                "column": col,
                "issue": "NULL value",
                "action": action
            })
    if action == "quarantine" and count:
        quarantine_mask = quarantine_mask.reindex(df_clean.index, fill_value=False) | mask

if quarantine_mask.any():
    df_quarantine = df_clean.loc[quarantine_mask].copy()
    df_quarantine["quarantine_reason"] = "NULL in configured column(s): " + df_quarantine.apply(
        lambda row: ", ".join(
            col for col, action in NULL_HANDLING_RULES.items()
            if action == "quarantine" and pd.isna(row[col])
        ),
        axis=1
    )
    quarantined_count = int(quarantine_mask.sum())
    df_clean = df_clean.loc[~quarantine_mask].copy()
    change_log.append({
        "column": ", ".join(col for col, action in NULL_HANDLING_RULES.items() if action == "quarantine"),
        "rule": "quarantine rows with configured NULLs",
        "rows_affected": quarantined_count,
        "reason": "Explicit NULL_HANDLING_RULES configuration; review quarantine output"
    })

issue_register = pd.DataFrame(
    mandatory_issues + null_rule_issues,
    columns=["row_index", "column", "issue", "action"]
)
display(issue_register)
print("Quarantined rows:", len(df_quarantine))
display(df_quarantine.head())

,row_index,column,issue,action


Quarantined rows: 0


,client_id,company_code,accounting_document_no,fiscal_year,line_item_no,account_type,debit_credit_indicator,amount_local_currency,amount_document_currency,assignment_no,line_item_text,payment_terms,cost_center_id,gl_account_no,customer_id,vendor_id,payement_baseline_date,tax_code,clearing_document_number,clearing_date,special_gl_indicator


## 10. Final validation

Check row counts, column counts, nulls, duplicates, and schema-level changes. Any row-count reduction must have an approved reason.


In [10]:
after_profile = profile_dataframe(df_clean)
validation_summary = {
    "source_table": SOURCE_TABLE,
    "target_table": TARGET_TABLE,
    "source_rows": int(len(df_raw)),
    "cleaned_rows": int(len(df_clean)),
    "quarantined_rows": int(len(df_quarantine)),
    "rows_removed_by_exact_deduplication": int(exact_duplicates_before if DROP_EXACT_DUPLICATES else 0),
    "rows_removed_to_quarantine": int(len(df_quarantine)),
    "rows_removed_total_from_clean_output": int(len(df_raw) - len(df_clean)),
    "source_columns": int(len(df_raw.columns)),
    "cleaned_columns": int(len(df_clean.columns)),
    "columns_excluded": ", ".join(COLUMNS_TO_DROP),
    "exact_duplicate_rows_before": int(df_raw.duplicated().sum()),
    "exact_duplicate_rows_after": int(df_clean.duplicated().sum()),
    "mandatory_issue_samples": int(len(mandatory_issues)),
    "null_rule_issue_samples": int(len(null_rule_issues)),
    "date_issue_samples": int(len(date_issues)),
    "dry_run": DRY_RUN,
    "write_enabled": WRITE_TO_DATABASE,
    "validated_at_utc": datetime.now(timezone.utc).isoformat()
}
display(pd.DataFrame([validation_summary]))
display(after_profile)

,source_table,target_table,source_rows,cleaned_rows,quarantined_rows,rows_removed_by_exact_deduplication,rows_removed_to_quarantine,rows_removed_total_from_clean_output,source_columns,cleaned_columns,columns_excluded,exact_duplicate_rows_before,exact_duplicate_rows_after,mandatory_issue_samples,null_rule_issue_samples,date_issue_samples,dry_run,write_enabled,validated_at_utc
0,stg_bseg,clean_bseg,332106,332029,0,77,0,77,21,20,tax_code,77,0,0,0,0,False,True,2026-10-22T16:30:05.568467+00:00


,dtype,null_count,null_pct,distinct_non_null,blank_string_count
special_gl_indicator,str,332024,99.998,1,0
line_item_text,str,331977,99.984,1,0
cost_center_id,str,331949,99.976,12,0
clearing_document_number,str,328605,98.969,1,0
clearing_date,str,328605,98.969,1,0
customer_id,str,293496,88.395,1,0
payment_terms,str,258027,77.712,1,0
payement_baseline_date,str,257694,77.612,1489,0
vendor_id,str,144722,43.587,28,0
assignment_no,str,134475,40.501,68695,0


## 11. Optional write to a separate cleaned table

**Keep `WRITE_TO_DATABASE = False` until you have reviewed the results.** This writes to the configured target table only, never to the staging table. It refuses to overwrite an existing target table so that previous outputs are not accidentally destroyed. Review table design and data types before using this generic `to_sql` write.


In [11]:
if WRITE_TO_DATABASE and not DRY_RUN:
    if inspector.has_table(TARGET_TABLE, schema=SCHEMA):
        raise RuntimeError(
            f"Target {SCHEMA}.{TARGET_TABLE} already exists. "
            "Choose a new target name or manage replacement explicitly."
        )
    with engine.begin() as conn:
        df_clean.to_sql(
            TARGET_TABLE,
            con=conn,
            schema=SCHEMA,
            if_exists="fail",
            index=False,
            method="multi",
            chunksize=5000
        )
    print(f"Wrote {len(df_clean):,} rows to {SCHEMA}.{TARGET_TABLE}")

    # Quarantine is exported to CSV by default; no separate database table is created.
    # This avoids silently creating additional tables without an explicit design decision.
else:
    print("No database write performed. Review validation_summary first.")

Wrote 332,029 rows to public.clean_bseg


## 12. Export evidence for the Sprint 2 report

Export the summary, column profile, change log, and issue samples for documentation. Avoid exporting sensitive data columns unnecessarily.


In [12]:
OUTPUT_DIR = Path("cleaning_evidence")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.DataFrame([baseline_summary]).to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_before_summary.csv", index=False)
baseline_profile.to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_before_profile.csv")
pd.DataFrame([validation_summary]).to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_validation_summary.csv", index=False)
after_profile.to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_after_profile.csv")
pd.DataFrame(change_log, columns=["column", "rule", "rows_affected", "reason"]).to_csv(
    OUTPUT_DIR / f"{SOURCE_TABLE}_change_log.csv", index=False
)
issue_register.to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_issues.csv", index=False)
pd.DataFrame(date_issues, columns=["row_index", "column", "original_value", "issue"]).to_csv(
    OUTPUT_DIR / f"{SOURCE_TABLE}_date_issues.csv", index=False
)
df_quarantine.to_csv(OUTPUT_DIR / f"{SOURCE_TABLE}_quarantine.csv", index=False)

print("Evidence exported to:", OUTPUT_DIR.resolve())
print("Quarantine CSV rows:", len(df_quarantine))

Evidence exported to: C:\Users\adity\Downloads\cleaning_evidence
Quarantine CSV rows: 0


## 13. Table-specific configuration worksheet

Complete this before running the template for each table.

| Item | Decision required |
|---|---|
| Source table | e.g. `stg_bkpf` |
| Target table | e.g. `clean_bkpf` |
| Candidate/business key | Confirm with data dictionary and source owner |
| Mandatory columns | Confirm business rules; do not infer solely from NULL percentages |
| Columns to drop | Add to `COLUMNS_TO_DROP` only after confirming they are unnecessary |
| Duplicate policy | `DROP_EXACT_DUPLICATES=True` only after investigating exact repeats; business-key duplicates are report-only |
| NULL policy | Configure `NULL_HANDLING_RULES` per column: `retain`, `flag`, or `quarantine` |
| Date/time columns | Inspect actual formats and invalid values |
| Flag columns | Confirm allowed values and whether text-to-boolean conversion is safe |
| Amount validation | Agree plausible limits and debit/credit sign conventions |
| Join validation | Check orphan keys only after key definitions are confirmed |

### EFDP-specific cautions from the existing reports
- `stg_skat.chart_of_accounts` is NULL in all profiled rows: do not invent a value. Keep it until the source/extraction question is resolved; if confirmed unnecessary, add it to `COLUMNS_TO_DROP` for the cleaned output.
- `stg_bseg.tax_code` is NULL in all profiled rows: confirm whether extraction omitted it or the field is unused.
- Very large BSEG amounts need business validation; do not delete them using an arbitrary threshold.
- `client_id` types differ across tables; standardize in the cleaned layer only after checking identifier formats and leading zeros.
- `is_deleted` is text in T002 but boolean in SKA1/SKAT; inspect distinct values before converting.
- The staging field `payement_baseline_date` is misspelled; rename only in the cleaned layer after confirming the intended target name.
- The profiling report did not validate candidate keys or date formats, so this template adds those checks as configurable steps.

### Example configuration for `stg_skat` (review only; do not enable blindly)
```python
SOURCE_TABLE = "stg_skat"
TARGET_TABLE = "clean_skat"
COLUMNS_TO_DROP = []  # Keep chart_of_accounts pending source-owner confirmation
DROP_EXACT_DUPLICATES = False  # Set True only after duplicate investigation
KEY_COLUMNS = ["client_id", "language_code", "chart_of_accounts", "gl_account_number"]  # verify before use
NULL_HANDLING_RULES = {
    "gl_account_description": "flag",
    "gl_account_short_name": "flag",
}
```
Note: with `chart_of_accounts` fully NULL, it is not suitable as a reliable key component until its origin is understood.